# 04 統計分析：多数作品を用いた本分析

`game_decline_summary.csv` と `review_summary_all.csv` をゲーム（appid）単位で結合し、カテゴリ差、ピーク規模、累積低評価率と衰退率の関連を検討する。

主要な3研究質問は内容が異なるため、生のp値を基本として報告する。一方、3検定全体を一つの探索的ファミリーとみなした場合のHolm補正p値も参考値として併記する。カテゴリ間の事後比較は同一研究質問内の多数のペア検定であるため、Holm補正を必須とする。


In [ ]:
# ライブラリと入出力先
from itertools import combinations
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.rcParams["font.family"] = ["Noto Sans CJK JP", "IPAexGothic", "Yu Gothic", "Hiragino Sans", "sans-serif"]
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ALPHA = 0.05
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/卒業研究/steam_research/data")
DATA_DIR = DRIVE_DATA_DIR if DRIVE_DATA_DIR.exists() else Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"データ入出力先: {DATA_DIR.resolve()}")


In [ ]:
# 共通関数
def require_csv(filename, required_columns):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"入力CSVが見つかりません: {path}\n"
            "前段の収集・集計スクリプトを実行し、指定ファイルをdataディレクトリに配置してください。"
        )
    df = pd.read_csv(path, encoding="utf-8-sig")
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{filename}に必要な列がありません: {missing}")
    return df


def holm_adjust(p_values):
    """NaNを保ったままHolm法でp値を補正する。"""
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(values), np.nan)
    valid_indices = np.flatnonzero(np.isfinite(values))
    if not len(valid_indices):
        return adjusted
    order = valid_indices[np.argsort(values[valid_indices])]
    m = len(order)
    running_max = 0.0
    for rank, index in enumerate(order):
        running_max = max(running_max, (m - rank) * values[index])
        adjusted[index] = min(running_max, 1.0)
    return adjusted


def rho_strength(rho):
    """Spearman rhoの大きさを控えめな目安で表す。"""
    if pd.isna(rho):
        return "判定不能"
    value = abs(rho)
    if value < 0.1:
        return "ほぼなし"
    if value < 0.3:
        return "弱い"
    if value < 0.5:
        return "中程度"
    return "強い"


def direction_text(value):
    if pd.isna(value) or value == 0:
        return "明確な方向なし"
    return "正" if value > 0 else "負"

print("注: 相関係数の強さの区分は解釈の目安であり、絶対的な基準ではありません。")


In [ ]:
# 集計済み衰退指標とレビュー概要を読み込み、appidで結合する
decline_columns = [
    "appid", "name", "category", "first_month", "latest_month", "peak_month",
    "months_observed", "peak_avg_players", "latest_avg_players", "decline_rate",
    "largest_monthly_drop_rate", "peak_to_latest_months",
]
review_columns = [
    "appid", "total_positive", "total_negative", "total_reviews", "negative_rate"
]
decline_df = require_csv("game_decline_summary.csv", decline_columns)
review_df = require_csv("review_summary_all.csv", review_columns)

for frame in (decline_df, review_df):
    frame["appid"] = pd.to_numeric(frame["appid"], errors="coerce").astype("Int64")

# 重複は品質チェックで報告する。結合の行数増加を防ぐため、分析用には末尾行を採用する。
decline_duplicate_count = int(decline_df["appid"].duplicated(keep=False).sum())
review_duplicate_count = int(review_df["appid"].duplicated(keep=False).sum())
decline_unique = decline_df.drop_duplicates("appid", keep="last").copy()
review_unique = review_df.drop_duplicates("appid", keep="last")[review_columns].copy()
analysis_df = decline_unique.merge(review_unique, on="appid", how="left", validate="one_to_one")

numeric_columns = [
    "months_observed", "peak_avg_players", "latest_avg_players", "decline_rate",
    "largest_monthly_drop_rate", "peak_to_latest_months", "total_positive",
    "total_negative", "total_reviews", "negative_rate",
]
for column in numeric_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")
for column in ["first_month", "latest_month", "peak_month"]:
    analysis_df[column] = pd.to_datetime(analysis_df[column], errors="coerce")

analysis_columns = decline_columns + ["total_positive", "total_negative", "total_reviews", "negative_rate"]
analysis_df = analysis_df[analysis_columns].sort_values(["category", "appid"]).reset_index(drop=True)
analysis_df.to_csv(DATA_DIR / "analysis_dataset.csv", index=False, encoding="utf-8-sig")

review_matched = int(analysis_df["total_reviews"].notna().sum())
review_missing = len(analysis_df) - review_matched
print(f"分析対象ゲーム数: {analysis_df['appid'].nunique()}")
print(f"カテゴリ数: {analysis_df['category'].nunique(dropna=True)}")
print("\nカテゴリ別作品数:")
for category, count in analysis_df["category"].value_counts(dropna=False).items():
    print(f"  {category}: {count}")
print(f"\nレビュー結合成功数: {review_matched}")
print(f"レビュー欠損数: {review_missing}")
print(f"分析データ保存完了: {DATA_DIR / 'analysis_dataset.csv'}")


In [ ]:
# 統計分析前のデータ品質チェック
quality_counts = {
    "decline入力のappid重複行数": decline_duplicate_count,
    "review入力のappid重複行数": review_duplicate_count,
    "analysis_dfのappid重複数": int(analysis_df["appid"].duplicated().sum()),
    "decline_rate欠損数": int(analysis_df["decline_rate"].isna().sum()),
    "negative_rate欠損数": int(analysis_df["negative_rate"].isna().sum()),
    "peak_avg_players欠損数": int(analysis_df["peak_avg_players"].isna().sum()),
    "category欠損数": int(analysis_df["category"].isna().sum()),
}
quality_df = pd.DataFrame({"check": quality_counts.keys(), "count": quality_counts.values()})
display(quality_df)

category_counts = analysis_df["category"].value_counts(dropna=False).rename("n").to_frame()
print("各カテゴリ作品数")
display(category_counts)
print("months_observedの要約統計")
display(analysis_df["months_observed"].describe().to_frame().T)
print("total_reviewsの要約統計")
display(analysis_df["total_reviews"].describe().to_frame().T)

for check, count in quality_counts.items():
    if count > 0:
        warnings.warn(f"データ品質警告: {check} = {count}")
for category, count in analysis_df["category"].value_counts(dropna=False).items():
    if count < 5:
        warnings.warn(f"標本数警告: カテゴリ「{category}」はn={count}（5未満）です。")
if analysis_df.empty:
    raise ValueError("結合後のanalysis_dfが空です。入力CSVのappidと内容を確認してください。")


In [ ]:
# 分析1: カテゴリ間の衰退率差（Kruskal–Wallis検定）
genre_data = analysis_df.dropna(subset=["category", "decline_rate"]).copy()
genre_summary = genre_data.groupby("category")["decline_rate"].agg(n="count", median="median", mean="mean")
genre_summary["IQR"] = genre_data.groupby("category")["decline_rate"].quantile(0.75) - genre_data.groupby("category")["decline_rate"].quantile(0.25)
genre_summary = genre_summary.sort_values("median")
display(genre_summary)

valid_groups = [(category, group["decline_rate"].to_numpy()) for category, group in genre_data.groupby("category") if len(group) > 0]
if len(valid_groups) < 2:
    raise ValueError("Kruskal–Wallis検定には、衰退率が有効なカテゴリが2つ以上必要です。")
kw_h, kw_p = stats.kruskal(*(values for _, values in valid_groups))
kw_k = len(valid_groups)
kw_n = sum(len(values) for _, values in valid_groups)
kw_epsilon_sq = max(0.0, (kw_h - kw_k + 1) / (kw_n - kw_k)) if kw_n > kw_k else np.nan

print("何を検定したか: カテゴリ間で衰退率の分布が異なるかをKruskal–Wallis検定で検討した。")
print("帰無仮説: すべてのカテゴリで衰退率の分布は同じ。")
print(f"H統計量: {kw_h:.4f}")
print(f"自由度: {kw_k - 1}")
print(f"p値: {kw_p:.6g}")
print(f"効果量 epsilon squared: {kw_epsilon_sq:.4f}")
print("解釈: " + ("5%水準で有意であり、少なくとも一部カテゴリの分布が異なることが示唆された。" if kw_p < ALPHA else "5%水準で有意ではなく、カテゴリ差を示す十分な証拠は得られなかった。"))
print("言えること: 観測標本におけるカテゴリ間の衰退率分布の差について順位に基づき評価できる。")
print("言えないこと: 有意差だけから全カテゴリ間の差、因果関係、Steamゲーム全体への一般化は断定できない。")

order = genre_summary.index.tolist()
plot_values = [genre_data.loc[genre_data["category"] == category, "decline_rate"].to_numpy() for category in order]
fig, ax = plt.subplots(figsize=(max(10, len(order) * 1.1), 6))
ax.boxplot(plot_values, tick_labels=order, showmeans=True)
rng = np.random.default_rng(42)
for position, values in enumerate(plot_values, start=1):
    ax.scatter(rng.normal(position, 0.045, len(values)), values, alpha=0.55, s=22, color="tab:blue")
ax.set(xlabel="カテゴリ", ylabel="ピークから最新月までの衰退率", title="カテゴリ別の衰退率")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Kruskal–Wallis検定が有意な場合のみ、Mann–Whitney U検定とHolm補正を行う
posthoc_columns = ["category_1", "category_2", "U", "raw_p", "holm_p", "significant"]
if kw_p < ALPHA:
    rows = []
    for (category_1, values_1), (category_2, values_2) in combinations(valid_groups, 2):
        u_stat, raw_p = stats.mannwhitneyu(values_1, values_2, alternative="two-sided", method="auto")
        rows.append({"category_1": category_1, "category_2": category_2, "U": u_stat, "raw_p": raw_p})
    posthoc_df = pd.DataFrame(rows)
    posthoc_df["holm_p"] = holm_adjust(posthoc_df["raw_p"])
    posthoc_df["significant"] = posthoc_df["holm_p"] < ALPHA
    posthoc_df = posthoc_df[posthoc_columns].sort_values("holm_p")
    display(posthoc_df)
    posthoc_df.to_csv(DATA_DIR / "posthoc_genre_results.csv", index=False, encoding="utf-8-sig")
    print(f"保存完了: {DATA_DIR / 'posthoc_genre_results.csv'}")
else:
    posthoc_df = pd.DataFrame(columns=posthoc_columns)
    print("Kruskal–Wallis検定が有意でなかったため、探索的な事後比較は実施しない")


In [ ]:
# 分析2: ピーク時プレイヤー規模と衰退率（Spearman順位相関）
peak_part = analysis_df.dropna(subset=["peak_avg_players", "decline_rate"]).copy()
# 対数軸に表示できる正の値だけを図示する。相関計算は欠損を除いた元の値を使用する。
if len(peak_part) < 3 or peak_part["peak_avg_players"].nunique() < 2 or peak_part["decline_rate"].nunique() < 2:
    peak_rho, peak_p = np.nan, np.nan
    warnings.warn("ピーク規模のSpearman相関に必要な標本数または変動が不足しています。")
else:
    peak_rho, peak_p = stats.spearmanr(peak_part["peak_avg_players"], peak_part["decline_rate"])

print(f"rho: {peak_rho:.4f}")
print(f"p値: {peak_p:.6g}")
print(f"n: {len(peak_part)}")
print(f"関連の方向: {direction_text(peak_rho)}")
print(f"効果の大きさ: {rho_strength(peak_rho)}（目安であり絶対的基準ではない）")
print("言えること: 観測標本内のピーク時プレイヤー規模と衰退率の単調な関連を評価できる。")
print("言えないこと: 相関から、規模が衰退を引き起こしたという因果関係は断定できない。")

plot_peak = peak_part[peak_part["peak_avg_players"] > 0]
plt.scatter(plot_peak["peak_avg_players"], plot_peak["decline_rate"], alpha=0.6)
plt.xscale("log")
plt.xlabel("ピーク時平均プレイヤー数（対数軸）")
plt.ylabel("衰退率")
plt.title("ピーク時プレイヤー規模と衰退率")
plt.tight_layout()
plt.show()


In [ ]:
# 分析3: 累積低評価率と衰退率（Spearman順位相関）
# レビュー0件の作品は除外する。
review_part = analysis_df.dropna(subset=["negative_rate", "decline_rate", "total_reviews"]).copy()
review_part = review_part[review_part["total_reviews"] > 0]
if len(review_part) < 3 or review_part["negative_rate"].nunique() < 2 or review_part["decline_rate"].nunique() < 2:
    review_rho, review_p = np.nan, np.nan
    warnings.warn("低評価率のSpearman相関に必要な標本数または変動が不足しています。")
else:
    review_rho, review_p = stats.spearmanr(review_part["negative_rate"], review_part["decline_rate"])

print(f"rho: {review_rho:.4f}")
print(f"p値: {review_p:.6g}")
print(f"n: {len(review_part)}")
print(f"関連の方向: {direction_text(review_rho)}")
print(f"効果の大きさ: {rho_strength(review_rho)}（目安であり絶対的基準ではない）")
print("言えること: 観測標本内の累積低評価率と衰退率の単調な関連を評価できる。")
print("言えないこと: 累積レビューの時間的前後関係は不明であり、相関から因果関係は断定できない。")

plt.scatter(review_part["negative_rate"], review_part["decline_rate"], alpha=0.6)
plt.xlabel("Steam累積レビューの低評価率")
plt.ylabel("衰退率")
plt.title("累積低評価率と衰退率")
plt.tight_layout()
plt.show()


In [ ]:
# 主要3研究質問の結果と、探索的なHolm補正p値を保存する
raw_p_values = np.array([kw_p, peak_p, review_p], dtype=float)
exploratory_holm = holm_adjust(raw_p_values)
results_df = pd.DataFrame([
    {
        "analysis": "カテゴリ間の衰退率差（Kruskal–Wallis）", "statistic": "H",
        "statistic_value": kw_h, "raw_p": kw_p, "effect_size": kw_epsilon_sq, "n": kw_n,
    },
    {
        "analysis": "ピーク時プレイヤー規模と衰退率（Spearman）", "statistic": "rho",
        "statistic_value": peak_rho, "raw_p": peak_p, "effect_size": abs(peak_rho), "n": len(peak_part),
    },
    {
        "analysis": "累積低評価率と衰退率（Spearman）", "statistic": "rho",
        "statistic_value": review_rho, "raw_p": review_p, "effect_size": abs(review_rho), "n": len(review_part),
    },
])
results_df["holm_p_exploratory"] = exploratory_holm
results_df["significant_raw"] = results_df["raw_p"] < ALPHA
results_df["significant_holm"] = results_df["holm_p_exploratory"] < ALPHA
results_df = results_df[[
    "analysis", "statistic", "statistic_value", "raw_p", "holm_p_exploratory",
    "effect_size", "n", "significant_raw", "significant_holm",
]]
display(results_df)
results_df.to_csv(DATA_DIR / "statistical_results.csv", index=False, encoding="utf-8-sig")
print("主要な3研究質問では生のp値を基本とし、研究質問をまたぐHolm補正値は探索的参考値として示します。")
print(f"保存完了: {DATA_DIR / 'statistical_results.csv'}")


In [ ]:
# 卒論本文用の自動文章
def p_interpretation(p_value, significant_text, nonsignificant_text):
    if pd.isna(p_value):
        return "データ不足のため検定結果を算出できなかった。"
    return significant_text if p_value < ALPHA else nonsignificant_text

print("## 結果に書ける内容")
print(f"- 分析対象は{analysis_df['appid'].nunique()}作品、{analysis_df['category'].nunique(dropna=True)}カテゴリであった。")
print(f"- カテゴリ間比較は H({kw_k - 1})={kw_h:.3f}, p={kw_p:.4g}, epsilon squared={kw_epsilon_sq:.3f} であり、" + p_interpretation(kw_p, "カテゴリ間で衰退率分布が異なることが示唆された。", "カテゴリ差を示す十分な証拠は得られなかった。ただし、これは『差がない』ことの証明ではない。"))
print(f"- ピーク時プレイヤー規模と衰退率の関連は rho={peak_rho:.3f}, p={peak_p:.4g}, n={len(peak_part)} で、方向は{direction_text(peak_rho)}、大きさの目安は{rho_strength(peak_rho)}であった。")
print(f"- 累積低評価率と衰退率の関連は rho={review_rho:.3f}, p={review_p:.4g}, n={len(review_part)} で、方向は{direction_text(review_rho)}、大きさの目安は{rho_strength(review_rho)}であった。")
print("- 主要3検定は生のp値を基本に報告し、3研究質問全体へのHolm補正p値を探索的参考値として併記した。")

print("\n## 考察に書ける内容")
print("- 有意な結果が得られた場合も、観察研究の相関やカテゴリ差から因果関係を断定することはできない。")
print("- 有意でない結果は『差や関連が存在しない』ことを意味せず、効果量、標本数、測定誤差を含めて解釈する必要がある。")
print("- Steamレビューは取得時点までの累積値であるため、低評価とプレイヤー衰退の時間的前後関係は不明である。")
print("- Spearman rhoの強さの区分は便宜的な目安であり、研究文脈と不確実性を踏まえる必要がある。")

print("\n## 研究上の限界")
limitations = [
    "decline_rateは観測期間中ピークから最新月までの値であり、ピークを事後選択することによるバイアスがある。",
    "発売時期と運営期間がゲーム間で異なり、観測期間も完全には統一されていない。",
    "SteamChartsに掲載され、かつデータを取得できた作品だけを対象とする選択バイアスがある。",
    "Steam Storeタグを利用したカテゴリ分類には、カテゴリ間の重複と意味の曖昧さがある。",
    "累積レビューのため、評価と衰退の時間的前後関係が不明である。",
    "未調整の交絡要因があり、相関から因果を断定できない。",
    "収集可能な標本に基づくため、結果をSteamゲーム全体へ一般化する際は注意が必要である。",
]
for limitation in limitations:
    print(f"- {limitation}")
